## Data Ingestion 
This notebook is used to ingest data from various sources and prepare it for use in a retrieval-augmented generation (RAG) system. The notebook includes code to load documents, split them into chunks, and create embeddings for the chunks using a language model. The embeddings are then stored in a vector database for later retrieval.

In [1]:
import langchain
import os
from typing import List, Dict, Any
import pandas as pd

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print("Langchain version:", langchain.__version__)

Langchain version: 1.2.0


### Understanding the document structure in Langchain

In [3]:
doc = Document(
    page_content="This is the main text content of the document.",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "John Doe",
        "date_created": "2024-06-01",
        "custom_field": "custom_value"
    }
)

print("Document content:", doc.page_content)
print("Document metadata:", doc.metadata)

Document content: This is the main text content of the document.
Document metadata: {'source': 'example.txt', 'page': 1, 'author': 'John Doe', 'date_created': '2024-06-01', 'custom_field': 'custom_value'}


In [4]:
type(doc)

langchain_core.documents.base.Document

### Reading text files from a directory

In [5]:
# Creating a simple .txt file for demonstration
os.makedirs("data/text_files", exist_ok=True)

In [6]:
sample_texts = {
    "data/text_files/python_intro.txt": """
Python is a high-level, interpreted programming language known for its readability and versatility. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python has a large standard library and a vibrant ecosystem of third-party packages, making it suitable for a wide range of applications such as web development, data analysis, artificial intelligence, scientific computing, and more.
Its syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to languages like C++ or Java. Python's dynamic typing and automatic memory management further enhance its ease of use. The language's popularity has surged in recent years, driven by its role in data science, machine learning, and automation tasks.
Some key features of Python include its extensive standard library, support for multiple programming paradigms, and a large community that contributes to its growth and development. Overall, Python's simplicity and power make it a preferred choice for both beginners and experienced developers.
Here are some basic examples of Python code:
# Hello World
print("Hello, World!")
# Simple Function
def greet(name):
    return f"Hello, {name}!"
print(greet("Alice"))
# List Comprehension
squares = [x**2 for x in range(10)]
print(squares)
""",

    "data/text_files/data_science_overview.txt": """
Data science is an interdisciplinary field that combines statistical analysis, computer science, and domain expertise to extract insights and knowledge from structured and unstructured data. It involves various stages, including data collection, cleaning, exploration, modeling, and visualization.
The primary goal of data science is to make informed decisions and predictions based on data analysis. Data scientists use a variety of tools and techniques, such as machine learning algorithms, statistical models, and data visualization tools, to analyze large datasets and uncover patterns and trends.
Data science has applications across numerous industries, including finance, healthcare, marketing, and technology. It plays a crucial role in driving business strategies, improving customer experiences, and optimizing operations.
Some key components of data science include:
1. Data Collection: Gathering data from various sources, such as databases, APIs, and web scraping.
2. Data Cleaning: Preprocessing and transforming raw data to ensure its quality and consistency.
3. Data Exploration: Analyzing data to understand its structure, relationships, and patterns.
4. Modeling: Building predictive models using machine learning algorithms to make forecasts and classifications.
5. Visualization: Creating visual representations of data to communicate findings effectively.
Overall, data science is a rapidly evolving field that continues to grow in importance as organizations increasingly rely
on data-driven decision-making.
"""
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created.")

Sample text files created.


In [7]:
# TextLoaders to read text files from a directory
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path="data/text_files/data_science_overview.txt", encoding="utf-8")

documents = loader.load()
print(type(documents))
print(documents)

<class 'list'>
[Document(metadata={'source': 'data/text_files/data_science_overview.txt'}, page_content='\nData science is an interdisciplinary field that combines statistical analysis, computer science, and domain expertise to extract insights and knowledge from structured and unstructured data. It involves various stages, including data collection, cleaning, exploration, modeling, and visualization.\nThe primary goal of data science is to make informed decisions and predictions based on data analysis. Data scientists use a variety of tools and techniques, such as machine learning algorithms, statistical models, and data visualization tools, to analyze large datasets and uncover patterns and trends.\nData science has applications across numerous industries, including finance, healthcare, marketing, and technology. It plays a crucial role in driving business strategies, improving customer experiences, and optimizing operations.\nSome key components of data science include:\n1. Data Col

In [8]:
print("Number of documents loaded:", len(documents))
print("First document content:", documents[0].page_content[:500])  # Print first 500 characters
print("First document metadata:", documents[0].metadata)


Number of documents loaded: 1
First document content: 
Data science is an interdisciplinary field that combines statistical analysis, computer science, and domain expertise to extract insights and knowledge from structured and unstructured data. It involves various stages, including data collection, cleaning, exploration, modeling, and visualization.
The primary goal of data science is to make informed decisions and predictions based on data analysis. Data scientists use a variety of tools and techniques, such as machine learning algorithms, statis
First document metadata: {'source': 'data/text_files/data_science_overview.txt'}


In [10]:
# DirectoryLoader to load all text files from a directory
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = dir_loader.load()
print(type(documents))

for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print("Content:", doc.page_content[:200])  # Print first 200 characters
    print("Metadata:", doc.metadata)
print("Number of documents loaded:", len(documents))

<class 'list'>

Document 1:
Content: 
Data science is an interdisciplinary field that combines statistical analysis, computer science, and domain expertise to extract insights and knowledge from structured and unstructured data. It invol
Metadata: {'source': 'data\\text_files\\data_science_overview.txt'}

Document 2:
Content: 
Python is a high-level, interpreted programming language known for its readability and versatility. It supports multiple programming paradigms, including procedural, object-oriented, and functional p
Metadata: {'source': 'data\\text_files\\python_intro.txt'}
Number of documents loaded: 2


### Text Splitting Strategies

In [14]:
print(documents)

[Document(metadata={'source': 'data\\text_files\\data_science_overview.txt'}, page_content='\nData science is an interdisciplinary field that combines statistical analysis, computer science, and domain expertise to extract insights and knowledge from structured and unstructured data. It involves various stages, including data collection, cleaning, exploration, modeling, and visualization.\nThe primary goal of data science is to make informed decisions and predictions based on data analysis. Data scientists use a variety of tools and techniques, such as machine learning algorithms, statistical models, and data visualization tools, to analyze large datasets and uncover patterns and trends.\nData science has applications across numerous industries, including finance, healthcare, marketing, and technology. It plays a crucial role in driving business strategies, improving customer experiences, and optimizing operations.\nSome key components of data science include:\n1. Data Collection: Gath

In [16]:
# Method 1: Character Text Splitter
text = documents[0].page_content
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=50,
    length_function=len
)

char_chunks = char_splitter.split_text(text)
print("Character Text Splitter Chunks:")
for i, chunk in enumerate(char_chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)
print("Total chunks created:", len(char_chunks))

Created a chunk of size 297, which is longer than the specified 200
Created a chunk of size 303, which is longer than the specified 200
Created a chunk of size 230, which is longer than the specified 200


Character Text Splitter Chunks:

Chunk 1:
Data science is an interdisciplinary field that combines statistical analysis, computer science, and domain expertise to extract insights and knowledge from structured and unstructured data. It involves various stages, including data collection, cleaning, exploration, modeling, and visualization.

Chunk 2:
The primary goal of data science is to make informed decisions and predictions based on data analysis. Data scientists use a variety of tools and techniques, such as machine learning algorithms, statistical models, and data visualization tools, to analyze large datasets and uncover patterns and trends.

Chunk 3:
Data science has applications across numerous industries, including finance, healthcare, marketing, and technology. It plays a crucial role in driving business strategies, improving customer experiences, and optimizing operations.

Chunk 4:
Some key components of data science include:
1. Data Collection: Gathering data from various so

In [ ]:
# Method 2: Recursive Character Text Splitter
# Using multiple separators for more intelligent splitting

simple_text = """LangChain is a powerful framework for building applications with large language models (LLMs). It provides tools and abstractions to simplify the process of integrating LLMs into various applications, such as chatbots, question-answering systems, and more. Another key feature of LangChain is its ability to manage and utilize external data sources, allowing developers to create more context-aware and dynamic applications. With LangChain, developers can easily chain together different components, such as prompt templates, memory management, and output parsers, to create complex workflows that leverage the capabilities of LLMs effectively.
This modular approach not only enhances the functionality of applications but also promotes code reusability and maintainability. Overall, LangChain serves as a comprehensive toolkit for developers looking to harness the power of large language models in their projects.
It offers a range of features, including prompt engineering, memory management, and integration with various data sources, making it easier to build sophisticated AI-driven applications. By abstracting away much of the complexity involved in working with LLMs, LangChain enables developers to focus on creating innovative solutions that leverage the strengths of these models.

Here are some examples of how LangChain can be used in different applications:"""
rec_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=200,
    chunk_overlap=50,
    length_function=len
)

rec_chunks = rec_splitter.split_text(simple_text)
print("Recursive Character Text Splitter Chunks:")
for i, chunk in enumerate(rec_chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)
print("Total chunks created:", len(rec_chunks))

Recursive Character Text Splitter Chunks:

Chunk 1:
LangChain is a powerful framework for building applications with large language models (LLMs). It provides tools and abstractions to simplify the process of integrating LLMs into various applications,

Chunk 2:
of integrating LLMs into various applications, such as chatbots, question-answering systems, and more. Another key feature of LangChain is its ability to manage and utilize external data sources,

Chunk 3:
to manage and utilize external data sources, allowing developers to create more context-aware and dynamic applications. With LangChain, developers can easily chain together different components, such

Chunk 4:
easily chain together different components, such as prompt templates, memory management, and output parsers, to create complex workflows that leverage the capabilities of LLMs effectively.

Chunk 5:
This modular approach not only enhances the functionality of applications but also promotes code reusability and maintaina

: 

### Note about TokenTextSplitter
The `TokenTextSplitter` is particularly useful when working with language models that have token limits. It splits text based on token counts rather than character counts, which helps ensure that the chunks fit within the model's constraints. This is especially important for models like GPT, where the number of tokens directly affects performance and cost. By using `TokenTextSplitter`, you can create more efficient and effective text chunks for embedding and retrieval tasks.